# Clasificador de piezas — entrenamiento intensivo (cámara de costado)

Entrena el modelo que dice **qué pieza** hay en cada casilla (13 clases: 6 piezas × 2 colores + vacía) con las fotos del
**entrenamiento de piezas** de la app. Está pensado para que funcione bien con **una sola cámara de costado** (la principal), aunque se
entrena también con las fotos de la otra cámara como datos extra.

**Qué hace distinto del entrenamiento local:**

- Usa una red **preentrenada en ImageNet** (EfficientNet-B0, ResNet-18 o ConvNeXt-Tiny) en lugar de la CNN chica de 3 capas: con pocas fotos
  por pieza (30–100) es lo que más precisión da.
- **Aumentaciones de ángulo**: perspectiva, rotación, inclinación, escala y recorte al azar, para que no dependa del ángulo exacto de la
  cámara. El color casi no se toca (el tono, nada): el blanco/negro de la pieza y la **marca rojiza de los alfiles** son pistas importantes.
- Separa entrenamiento / validación / prueba **por captura** (las dos fotos de un mismo instante quedan del mismo lado), y mide la precisión
  **solo con la cámara de costado**, que es la que importa.
- Exporta un `clasificador_piezas.pt` que el backend carga tal cual (TorchScript con la normalización adentro: no hace falta instalar nada).

**Cómo usarlo:**

1. En la app, *Cámara → Piezas → Exportar dataset*: baja `dataset_piezas.zip` (recortes justo, ampliado y volumen + `metadatos.csv`).
2. Subilo a tu Google Drive, en «Mi unidad», sin descomprimir (~130 MB). La celda 3 lo busca ahí.
3. *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*. Después *Ejecutar todas*.
4. Al final se descarga `clasificador_piezas.pt`. En la compu del backend: guardá una copia del actual
   (`training/checkpoints/clasificador_piezas.pt`), reemplazalo por el nuevo y **reiniciá el backend**.

## 1 · Configuración

In [ ]:
RECORTE = "ampliado"            # "ampliado" (casilla + margen, recomendado), "volumen" (caja 3D: necesita el lente bien calibrado) o "justo"
CAMARA_OBJETIVO = "principal"   # la cámara con la que se va a usar el modelo (la de costado); con ella se valida y se mide
USAR_OTRAS_CAMARAS = True       # sumar al entrenamiento las fotos de la otra cámara (más datos y más variedad de ángulo)
PESO_OTRAS_CAMARAS = 0.5        # cuánto pesan esas fotos frente a las de la cámara objetivo
ARQUITECTURA = "efficientnet_b0"  # "efficientnet_b0", "resnet18" o "convnext_tiny" (más lenta, a veces mejor)
TAMANO = 128                    # lado de la imagen que entra a la red (el backend redimensiona igual)
EPOCAS = 60
PACIENCIA = 15                  # corta si la validación no mejora en tantas épocas
LOTE = 64
LR = 3e-4
SEMILLA = 7
FRACCION_VALIDACION = 0.15
FRACCION_PRUEBA = 0.15
RUTA_ZIP_EN_DRIVE = None  # None = usa el ZIP dataset_piezas*.zip MÁS NUEVO de tu Drive. Para forzar uno: "/content/drive/MyDrive/dataset_piezas_20261023_1950.zip"

CLASES = sorted([f"{c}-{p}" for c in ("white", "black") for p in ("pawn", "knight", "bishop", "rook", "queen", "king")]) + ["vacia"]
# mismo orden que el backend (modelo_piezas.CLASES)

## 2 · Entorno

In [ ]:
import os, io, json, random, zipfile, time, math
from collections import Counter, defaultdict
import numpy as np, pandas as pd, cv2, torch, torchvision
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.transforms import v2 as T
import matplotlib.pyplot as plt

EN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"  # mps: GPU de una Mac
print(f"torch {torch.__version__} · torchvision {torchvision.__version__} · dispositivo: {DISPOSITIVO}")
if DISPOSITIVO == "cpu":
    print("⚠ Sin GPU: va a ser MUY lento (horas). En Colab: Entorno de ejecución → Cambiar tipo de entorno → GPU; o corrélo en una Mac con chip Apple.")
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)

## 3 · Datos: el ZIP exportado desde la app

In [ ]:
CARPETA = "/content/dataset" if EN_COLAB else os.environ.get("CARPETA_DATASET", "dataset")
ruta_zip = os.environ.get("RUTA_ZIP")
if EN_COLAB and not ruta_zip:
    import glob
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)  # remontar: un archivo recién subido a Drive a veces no aparece hasta hacerlo
    mi_unidad = "/content/drive/MyDrive"
    candidatos = sorted(glob.glob(f"{mi_unidad}/dataset_piezas*.zip") + glob.glob(f"{mi_unidad}/*/dataset_piezas*.zip"), key=os.path.getmtime)
    for z in candidatos:
        print(f"  ZIP en tu Drive: {z.replace(mi_unidad + '/', '')} · {os.path.getsize(z) / 1e6:.0f} MB · {time.strftime('%d/%m %H:%M', time.localtime(os.path.getmtime(z)))}")
    if RUTA_ZIP_EN_DRIVE and os.path.exists(RUTA_ZIP_EN_DRIVE):
        ruta_zip = RUTA_ZIP_EN_DRIVE
    elif candidatos:
        ruta_zip = candidatos[-1]  # el más nuevo, aunque se llame «dataset_piezas (1).zip» o esté en una carpeta
    else:
        zips = sorted(os.path.basename(z) for z in glob.glob(f"{mi_unidad}/*.zip"))
        raise FileNotFoundError(
            "No encontré dataset_piezas.zip en tu Drive (en «Mi unidad» o en una carpeta de primer nivel). Subilo a drive.google.com, "
            "esperá a que termine de subir y volvé a ejecutar esta celda. ZIP que sí hay en Mi unidad: " + (", ".join(zips) or "ninguno")
        )
    print("Usando", ruta_zip, f"({os.path.getsize(ruta_zip) / 1e6:.0f} MB)")
if not os.path.isdir(os.path.join(CARPETA, "ampliado")):
    with zipfile.ZipFile(ruta_zip) as z:
        z.extractall(CARPETA)
datos = pd.read_csv(os.path.join(CARPETA, "metadatos.csv"), dtype=str).fillna("")
print(f"{len(datos)} recortes en total")
datos = datos[(datos.recorte == RECORTE) & datos.clase.isin(CLASES)].copy()
datos.loc[datos.grupo == "", "grupo"] = datos.loc[datos.grupo == "", "captura"]
datos["camara"] = datos.camara.replace("", "desconocida")
print(f"{len(datos)} recortes «{RECORTE}»")
display(pd.crosstab(datos.clase, datos.camara, margins=True)) if "display" in globals() else print(pd.crosstab(datos.clase, datos.camara))

## 4 · División por captura

Entrenamiento, validación y prueba se separan por **grupo** (el instante de la foto): las fotos de las dos cámaras de un mismo momento quedan
siempre del mismo lado, así la prueba es honesta. Validación y prueba usan **solo la cámara objetivo**.

In [ ]:
def dividir(datos):
    azar = random.Random(SEMILLA)
    clase_del_grupo = datos.groupby("grupo").clase.agg(lambda c: Counter(c).most_common(1)[0][0])
    destino = {}
    for clase, grupos in clase_del_grupo.groupby(clase_del_grupo):  # estratificado: cada clase reparte sus grupos
        lista = sorted(grupos.index); azar.shuffle(lista)
        n_prueba = max(1, round(len(lista) * FRACCION_PRUEBA)) if len(lista) >= 3 else 0
        n_valid = max(1, round(len(lista) * FRACCION_VALIDACION)) if len(lista) >= 3 else 0
        for i, g in enumerate(lista):
            destino[g] = "prueba" if i < n_prueba else "valid" if i < n_prueba + n_valid else "entreno"
    datos = datos.assign(parte=datos.grupo.map(destino))
    objetivo = datos.camara == CAMARA_OBJETIVO
    entreno = datos[(datos.parte == "entreno") & (objetivo | USAR_OTRAS_CAMARAS)]
    return entreno, datos[(datos.parte == "valid") & objetivo], datos[(datos.parte == "prueba") & objetivo]

entreno, valid, prueba = dividir(datos)
print(f"entrenamiento {len(entreno)} · validación {len(valid)} · prueba {len(prueba)} (las dos últimas solo «{CAMARA_OBJETIVO}»)")
print(pd.DataFrame({"entreno": entreno.clase.value_counts(), "valid": valid.clase.value_counts(), "prueba": prueba.clase.value_counts()}).fillna(0).astype(int))

## 5 · Imágenes y aumentaciones

Se lee igual que en el backend (`cv2`, BGR → RGB, redimensionado lineal, de 0 a 1). Las aumentaciones simulan otra posición de la cámara
(perspectiva, rotación, inclinación, escala, encuadre), otra luz (brillo y contraste) y desenfoque. **No** se cambia el tono ni se pasa a
gris: el color de la pieza y la marca rojiza de los alfiles tienen que llegar intactos.

In [ ]:
def leer(ruta):
    imagen = cv2.imread(os.path.join(CARPETA, ruta))
    imagen = cv2.resize(cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB), (TAMANO, TAMANO), interpolation=cv2.INTER_LINEAR)
    return torch.from_numpy(imagen).permute(2, 0, 1).float() / 255.0

AUMENTAR = T.Compose([
    T.RandomResizedCrop(TAMANO, scale=(0.72, 1.0), ratio=(0.85, 1.18), antialias=True),
    T.RandomPerspective(distortion_scale=0.35, p=0.6),
    T.RandomAffine(degrees=12, translate=(0.06, 0.06), shear=(-8, 8, -4, 4)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.2, hue=0.0),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 1.6))], p=0.3),
    T.RandomErasing(p=0.2, scale=(0.02, 0.08)),
])

class Recortes(Dataset):
    def __init__(self, filas, aumentar):
        self.rutas, self.y, self.aumentar = list(filas.archivo), [CLASES.index(c) for c in filas.clase], aumentar
        self.cache = {}
    def __len__(self):
        return len(self.rutas)
    def __getitem__(self, i):
        if i not in self.cache:
            self.cache[i] = leer(self.rutas[i])
        x = self.cache[i]
        return (AUMENTAR(x).clamp(0, 1) if self.aumentar else x), self.y[i]

def muestreador(filas):
    """Cada clase sale con la misma frecuencia; las fotos de la otra cámara pesan PESO_OTRAS_CAMARAS."""
    por_clase = filas.clase.value_counts()
    pesos = [(1.0 / por_clase[c]) * (1.0 if cam == CAMARA_OBJETIVO else PESO_OTRAS_CAMARAS) for c, cam in zip(filas.clase, filas.camara)]
    return WeightedRandomSampler(pesos, num_samples=len(filas), replacement=True)

trabajadores = 2 if EN_COLAB else 0
cargador = DataLoader(Recortes(entreno, True), batch_size=LOTE, sampler=muestreador(entreno), num_workers=trabajadores)
cargador_valid = DataLoader(Recortes(valid, False), batch_size=LOTE, num_workers=trabajadores)
cargador_prueba = DataLoader(Recortes(prueba, False), batch_size=LOTE, num_workers=trabajadores)

x, _ = next(iter(DataLoader(Recortes(entreno, True), batch_size=16, shuffle=True)))
plt.figure(figsize=(12, 3)); plt.imshow(torchvision.utils.make_grid(x, nrow=16).permute(1, 2, 0)); plt.axis("off"); plt.title("ejemplos aumentados"); plt.show()

## 6 · Modelo

In [ ]:
class Clasificador(nn.Module):
    """Red preentrenada con la normalización de ImageNet ADENTRO: recibe RGB de 0 a 1, como lo prepara el backend."""
    def __init__(self, red):
        super().__init__()
        self.red = red
        self.register_buffer("media", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("desvio", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x):
        return self.red((x - self.media) / self.desvio)

def crear_red(nombre, clases):
    if nombre == "resnet18":
        red = torchvision.models.resnet18(weights="DEFAULT"); red.fc = nn.Linear(red.fc.in_features, clases); cabeza = red.fc
    elif nombre == "efficientnet_b0":
        red = torchvision.models.efficientnet_b0(weights="DEFAULT"); red.classifier[1] = nn.Linear(red.classifier[1].in_features, clases); cabeza = red.classifier
    elif nombre == "convnext_tiny":
        red = torchvision.models.convnext_tiny(weights="DEFAULT"); red.classifier[2] = nn.Linear(red.classifier[2].in_features, clases); cabeza = red.classifier
    else:
        raise ValueError(nombre)
    return Clasificador(red), cabeza

modelo, cabeza = crear_red(ARQUITECTURA, len(CLASES))
modelo = modelo.to(DISPOSITIVO)
print(f"{ARQUITECTURA}: {sum(p.numel() for p in modelo.parameters()) / 1e6:.1f} M parámetros")

## 7 · Entrenamiento

AdamW con calentamiento y coseno, *label smoothing* y precisión mixta. Las 2 primeras épocas solo se entrena la cabeza (la parte nueva), así
no se rompe lo que la red ya sabe de ImageNet. Se guarda la época con mejor **F1 macro** en validación (pesa igual cada pieza, aunque haya
muchas más casillas vacías).

In [ ]:
def evaluar(red, cargador_eval, espejo=True):
    red.eval(); reales, probs = [], []
    with torch.no_grad(), torch.autocast(DISPOSITIVO, enabled=DISPOSITIVO == "cuda"):
        for x, y in cargador_eval:
            x = x.to(DISPOSITIVO)
            p = torch.softmax(red(x).float(), 1)
            if espejo:  # promedio con la imagen espejada (TTA)
                p = (p + torch.softmax(red(torch.flip(x, dims=[3])).float(), 1)) / 2
            probs.append(p.cpu()); reales += list(y)
    probs = torch.cat(probs).numpy() if probs else np.zeros((0, len(CLASES)))
    return np.array(reales), probs

def f1_macro(reales, predichas):
    f1s = []
    for k in range(len(CLASES)):
        vp = np.sum((predichas == k) & (reales == k)); fp = np.sum((predichas == k) & (reales != k)); fn = np.sum((predichas != k) & (reales == k))
        if vp + fn:
            f1s.append(2 * vp / max(1, 2 * vp + fp + fn))
    return float(np.mean(f1s)) if f1s else 0.0

optimizador = torch.optim.AdamW(modelo.parameters(), lr=LR, weight_decay=0.05 if ARQUITECTURA == "convnext_tiny" else 1e-4)
pasos = EPOCAS * max(1, len(cargador)); calentamiento = 2 * max(1, len(cargador))
planificador = torch.optim.lr_scheduler.LambdaLR(optimizador, lambda s: min(1, (s + 1) / calentamiento) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / pasos))))
perdida_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
escalador = torch.amp.GradScaler(enabled=DISPOSITIVO == "cuda")
historial, mejor, mejor_estado, sin_mejora = [], -1.0, None, 0
for epoca in range(1, EPOCAS + 1):
    solo_cabeza = epoca <= 2
    for nombre, p in modelo.red.named_parameters():
        p.requires_grad = (not solo_cabeza) or any(p is q for q in cabeza.parameters())
    modelo.train(); inicio = time.time(); suma, n = 0.0, 0
    for x, y in cargador:
        x, y = x.to(DISPOSITIVO), y.to(DISPOSITIVO)
        optimizador.zero_grad(set_to_none=True)
        with torch.autocast(DISPOSITIVO, enabled=DISPOSITIVO == "cuda"):
            perdida = perdida_fn(modelo(x), y)
        escalador.scale(perdida).backward(); escalador.step(optimizador); escalador.update(); planificador.step()
        suma += perdida.item() * len(y); n += len(y)
    reales, probs = evaluar(modelo, cargador_valid, espejo=False)
    exactitud = float((probs.argmax(1) == reales).mean()) if len(reales) else 0.0
    f1 = f1_macro(reales, probs.argmax(1))
    historial.append({"epoca": epoca, "perdida": suma / max(1, n), "exactitud": exactitud, "f1": f1})
    print(f"época {epoca:3d} · pérdida {suma / max(1, n):.3f} · validación: exactitud {exactitud:.3f}, F1 macro {f1:.3f} · {time.time() - inicio:.0f} s")
    if f1 > mejor:
        mejor, sin_mejora = f1, 0
        mejor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
    else:
        sin_mejora += 1
        if sin_mejora >= PACIENCIA:
            print(f"Sin mejora en {PACIENCIA} épocas: corto."); break
modelo.load_state_dict(mejor_estado)
h = pd.DataFrame(historial)
plt.figure(figsize=(10, 3)); plt.subplot(1, 2, 1); plt.plot(h.epoca, h.perdida); plt.title("pérdida (entrenamiento)")
plt.subplot(1, 2, 2); plt.plot(h.epoca, h.exactitud, label="exactitud"); plt.plot(h.epoca, h.f1, label="F1 macro"); plt.legend(); plt.title("validación"); plt.show()

## 8 · Prueba (fotos que el modelo nunca vio, solo cámara de costado)

In [ ]:
reales, probs = evaluar(modelo, cargador_prueba, espejo=True)
predichas = probs.argmax(1)
exactitud_prueba = float((predichas == reales).mean()) if len(reales) else float("nan")
f1_prueba = f1_macro(reales, predichas)
color = lambda k: CLASES[k].split("-")[0]
color_mal = int(sum(1 for r, p in zip(reales, predichas) if CLASES[r] != "vacia" and CLASES[p] != "vacia" and color(r) != color(p)))
print(f"Prueba ({len(reales)} recortes de «{CAMARA_OBJETIVO}»): exactitud {exactitud_prueba:.3f} · F1 macro {f1_prueba:.3f} · color equivocado: {color_mal}")
tabla = pd.DataFrame([{"clase": c, "fotos": int(np.sum(reales == k)), "aciertos": int(np.sum((reales == k) & (predichas == k)))} for k, c in enumerate(CLASES)])
tabla["precisión"] = (tabla.aciertos / tabla.fotos.clip(lower=1)).round(3)
print(tabla.to_string(index=False))
matriz = np.zeros((len(CLASES), len(CLASES)), int)
for r, p in zip(reales, predichas):
    matriz[r, p] += 1
plt.figure(figsize=(8, 7)); plt.imshow(matriz, cmap="Blues"); plt.xticks(range(len(CLASES)), CLASES, rotation=90); plt.yticks(range(len(CLASES)), CLASES)
for i in range(len(CLASES)):
    for j in range(len(CLASES)):
        if matriz[i, j]:
            plt.text(j, i, matriz[i, j], ha="center", va="center", color="white" if matriz[i, j] > matriz.max() / 2 else "black", fontsize=8)
plt.xlabel("predicha"); plt.ylabel("real"); plt.title("matriz de confusión (prueba)"); plt.tight_layout(); plt.show()
errores = [i for i in range(len(reales)) if reales[i] != predichas[i]][:24]
if errores:
    ds = Recortes(prueba, False)
    plt.figure(figsize=(16, 2.6 * math.ceil(len(errores) / 8)))
    for n, i in enumerate(errores):
        plt.subplot(math.ceil(len(errores) / 8), 8, n + 1); plt.imshow(ds[i][0].permute(1, 2, 0)); plt.axis("off")
        plt.title(f"{CLASES[reales[i]]}\n→ {CLASES[predichas[i]]}", fontsize=7)
    plt.suptitle("errores en prueba (real → predicha)"); plt.tight_layout(); plt.show()

## 9 · Exportar para el backend

Se guarda como **TorchScript** dentro del mismo formato de checkpoint que lee el backend (`backend/servicios/vision/piezas.py`): `clases`,
`tamano`, `recorte` y la red completa (con la normalización adentro). Antes de guardar se comprueba que la red exportada da lo mismo que la
original.

In [ ]:
modelo_cpu = modelo.float().cpu().eval()
ejemplo = torch.rand(8, 3, TAMANO, TAMANO)
# Se comprueba con FOTOS REALES de la prueba (el ruido al azar es un caso raro donde la red duda y cualquier diferencia mínima se amplifica).
# Se prueba primero la red optimizada (`freeze` funde capas y es más rápida); si no da lo mismo que la original, se usa la sin optimizar, que es idéntica.
reales_x = torch.stack([Recortes(prueba, False)[i][0] for i in range(min(48, len(prueba)))])
with torch.no_grad():
    trazada = torch.jit.trace(modelo_cpu, ejemplo[:2])
    p_original = torch.softmax(modelo_cpu(reales_x), 1)
    exportada = None
    for variante, red in (("optimizada", lambda: torch.jit.freeze(trazada)), ("sin optimizar", lambda: trazada)):
        candidata = red()
        p_exportada = torch.softmax(candidata(reales_x), 1)
        diferencia = (p_exportada - p_original).abs().max().item()
        igual_clase = bool((p_exportada.argmax(1) == p_original.argmax(1)).all())
        print(f"  comprobación de la exportación ({variante}): {len(reales_x)} fotos reales · probabilidades distintas hasta {diferencia:.4f} · misma clase en todas: {igual_clase}")
        if diferencia < 2e-2 and igual_clase:
            exportada = candidata
            break
assert exportada is not None, f"La red exportada no coincide con la original (última diferencia: {diferencia}, misma clase: {igual_clase})"
buffer = io.BytesIO(); torch.jit.save(exportada, buffer)
checkpoint = {
    "formato": "torchscript", "torchscript": buffer.getvalue(), "clases": CLASES, "tamano": TAMANO, "recorte": RECORTE,
    "arquitectura": ARQUITECTURA, "camara_objetivo": CAMARA_OBJETIVO, "torch": torch.__version__,
    "metricas": {"exactitud_prueba": exactitud_prueba, "f1_macro_prueba": f1_prueba, "f1_macro_validacion": mejor, "fotos_prueba": int(len(reales))},
    "fecha": time.strftime("%Y-%m-%d %H:%M"),
}
torch.save(checkpoint, "clasificador_piezas.pt")
json.dump({k: v for k, v in checkpoint.items() if k != "torchscript"}, open("clasificador_piezas.json", "w"), indent=2, ensure_ascii=False)
prueba.to_csv("prueba.csv", index=False)  # qué fotos fueron de prueba: para comparar con otro modelo sobre las mismas
print(f"Guardado clasificador_piezas.pt ({len(buffer.getvalue()) / 1e6:.1f} MB) · prueba: exactitud {exactitud_prueba:.3f}, F1 {f1_prueba:.3f}")
if EN_COLAB:
    from google.colab import files
    if os.path.isdir("/content/drive/MyDrive"):
        import shutil; shutil.copy("clasificador_piezas.pt", "/content/drive/MyDrive/clasificador_piezas.pt"); print("Copia en Drive: MyDrive/clasificador_piezas.pt")
    try:
        files.download("clasificador_piezas.pt")
    except Exception as error:  # desde VS Code la descarga directa no funciona: queda la copia en Drive
        print("No se pudo descargar directo (", error, "): bajalo de Drive, MyDrive/clasificador_piezas.pt")

## 10 · Instalarlo en el backend

1. Guardá una copia del modelo actual: `training/checkpoints/clasificador_piezas.pt` → por ejemplo `clasificador_piezas.anterior.pt`.
2. Copiá el `clasificador_piezas.pt` descargado en `training/checkpoints/`.
3. Reiniciá el backend. En *Probador* tiene que aparecer «piezas (recorte ampliado)» (o el que elegiste).

**Si querés probar otra variante**, cambiá `ARQUITECTURA` o `RECORTE` en la celda 1 y volvé a ejecutar todo; compará la exactitud y el F1 de
la prueba. `volumen` solo conviene cuando el lente de la cámara esté bien calibrado (si no, la caja 3D queda corrida).

El backend usa `torch 2.14`: un modelo exportado con una versión de torch igual o más vieja se carga sin problema (la versión queda anotada
en el checkpoint, campo `torch`).